# Haar Cascade ile Yüz ve Göz Algılama

Bu modül; Paul Viola ve Michael Jones tarafından 2001 yılında geliştirilen ve bilgisayarlı görüde gerçek zamanlı nesne algılamanın önünü açan **Viola-Jones Algoritması**'nı ve OpenCV'nin önceden eğitilmiş Haar Cascade modellerini inceler.

---

## 1. Viola-Jones Algoritmasının 4 Temel Direği

1. **Haar-Benzeri Öznitelikler (Haar-like Features):**
   - Dikdörtgen komşu bölgeler arasındaki piksel yoğunluk farkını hesaplar (Kenar öznitelikleri, Çizgi öznitelikleri, Dörtlü dikdörtgenler).
   - Örneğin bir yüzde göz bölgesi, yanaklara göre her zaman daha koyudur.

2. **İntegral Görüntü (Integral Image):**
   - Görüntüdeki herhangi bir dikdörtgen alanın piksel toplamını sadece 4 köşe referansı ile $O(1)$ sabit zamanda hesaplamayı sağlar. Bu sayede on binlerce öznitelik anlık olarak taranabilir.

3. **AdaBoost (Adaptive Boosting):**
   - $24 \times 24$ piksellik bir pencerede 160.000'den fazla olası Haar özniteliği arasından sınıflandırma için en kritik olan en iyi öznitelikleri (genellikle birkaç yüz adet) seçer.

4. **Basamaklı Sınıflandırıcı (Attentional Cascade):**
   - Sınıflandırıcılar kademeli (cascade) olarak sıralanır. Yüz içermeyen bir pencere ilk 1-2 kademede hemen elenir; sonraki karmaşık aşamalara sadece aday pencereler geçer. Bu sayede gerçek zamanlı FPS elde edilir.


In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

# 1. Önceden Eğitilmiş Modelleri Yükleme
face_cascade = cv2.CascadeClassifier('haarcascade_frontalface_default.xml')
eye_cascade = cv2.CascadeClassifier('haarcascade_eye.xml')

print("Face Cascade yüklendi:", not face_cascade.empty())
print("Eye Cascade yüklendi :", not eye_cascade.empty())

# Test görselini yükleme
image = cv2.imread('synthetic_face.png')
gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)


## 2. detectMultiScale Fonksiyonu ve Parametre Analizi

`face_cascade.detectMultiScale(image, scaleFactor, minNeighbors, minSize)`:
- `scaleFactor`: Görüntü piramidinde her adımda görselin küçültülme katsayısı (örneğin $1.1$, her adımda görseli %10 küçülterek farklı uzaklıktaki yüzleri arar).
- `minNeighbors`: Bir bölgenin yüz olarak kabul edilmesi için çevresinde bulunması gereken minimum aday dikdörtgen sayısı (Yanlış pozitifleri engeller).
- `minSize`: Algılanacak minimum yüz boyutu (örneğin $(30, 30)$ piksel altındaki detaylar elenir).


In [ ]:
# Yüz tespiti
faces = face_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=3, minSize=(60, 60))
print(f"Tespit Edilen Yüz Sayısı: {len(faces)}")

output = image.copy()

for (x, y, w, h) in faces:
    # Yüz sınırına yeşil çerçeve
    cv2.rectangle(output, (x, y), (x + w, y + h), (0, 255, 0), 3)
    cv2.putText(output, "Yuz", (x, y - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
    
    # İlgi Alanı (ROI - Region of Interest): Gözler yüzün içinde aranır
    roi_gray = gray[y:y+h, x:x+w]
    roi_color = output[y:y+h, x:x+w]
    
    eyes = eye_cascade.detectMultiScale(roi_gray, scaleFactor=1.1, minNeighbors=2, minSize=(15, 15))
    for (ex, ey, ew, eh) in eyes:
        # Göz sınırına mavi çerçeve
        cv2.rectangle(roi_color, (ex, ey), (ex + ew, ey + eh), (255, 0, 0), 2)

plt.figure(figsize=(8, 8))
plt.title("Haar Cascade Yüz (Yesil) ve Göz (Mavi) Tespiti")
plt.imshow(cv2.cvtColor(output, cv2.COLOR_BGR2RGB))
plt.axis('off')
plt.show()


## 3. Parametre Duyarlılığı ve Modern Alternatifler

- **Işık ve Açı Duyarlılığı:** Haar modelleri cepheden (frontal) ve iyi aydınlatılmış yüzlerde yüksek başarı gösterirken, yan profilde ve aşırı gölgede başarı oranı düşer.
- **Modern Bilgisayarlı Görü Evrimi:**
  1. Haar Cascade (2001) $\rightarrow$ CPU dostu, ultra hızlı.
  2. HOG + Linear SVM (Dlib) $\rightarrow$ Daha hassas kontur tabanlı.
  3. SSD / MTCNN / YOLO / MediaPipe $\rightarrow$ Derin öğrenme tabanlı, her açıdan ve karmaşık arka planlarda dayanıklı.
